# Preprocessing and cleaning

Archived from the former Python scripts/package so the Hermes experiments are kept as notebooks.


## Shared Project Paths

Path constants used by the pipeline.

_Archived from `src/icd10_coding/paths.py`._


In [ ]:
"""Project paths used by scripts and pipeline modules."""

from pathlib import Path


def find_project_root(start: Path | None = None) -> Path:
    """Find the repository root from a script, module, or notebook location."""
    current = (start or Path.cwd()).resolve()
    for path in (current, *current.parents):
        if (path / "data").exists() and (path / "src").exists():
            return path
    return current


PROJECT_ROOT = find_project_root()
DATA_RAW_DIR = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PREDICTIONS_DIR = PROJECT_ROOT / "output" / "predictions"
SUBMISSIONS_DIR = PROJECT_ROOT / "output" / "submissions"


def ensure_output_dirs() -> None:
    """Create generated-data directories used by the pipeline."""
    DATA_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
    PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)
    SUBMISSIONS_DIR.mkdir(parents=True, exist_ok=True)


## Text Normalisation Helpers

Cleaning utilities for accents, punctuation, casing, and whitespace.

_Archived from `src/icd10_coding/preprocessing/text.py`._


In [ ]:
"""Text cleaning used by the ICD-10 codification pipeline."""

import re
import unicodedata

import pandas as pd


def basic_clean(text: object) -> str:
    """Light cleaning that keeps literals readable."""
    if pd.isna(text):
        return ""
    text = str(text)
    text = re.sub(r"<[^>]+>", "", text)
    text = text.replace("`", "'").replace("Â´", "'")
    text = re.sub(r"\s+", " ", text).strip()
    return text


def normalize(text: object) -> str:
    """Aggressive normalized view used for matching and vectorization."""
    text = basic_clean(text).lower()
    text = unicodedata.normalize("NFD", text)
    text = "".join(c for c in text if unicodedata.category(c) != "Mn")
    return text.strip()


## Dataset Preparation Pipeline

Code for loading raw files, preparing labels, and exporting processed data.

_Archived from `src/icd10_coding/processing/preprocess.py`._


In [ ]:
"""Step 0: clean raw Kaggle files and enrich the training data."""

import pandas as pd

from icd10_coding.paths import DATA_PROCESSED_DIR, DATA_RAW_DIR, ensure_output_dirs
from icd10_coding.preprocessing.text import basic_clean, normalize


def run() -> None:
    ensure_output_dirs()

    train = pd.read_csv(DATA_RAW_DIR / "codification_data.csv")
    test = pd.read_csv(DATA_RAW_DIR / "leaderboard_data.csv")
    reference = pd.read_csv(DATA_RAW_DIR / "icd_d_p_pairs.csv")

    print(f"Training:  {len(train):,} rows, columns: {list(train.columns)}")
    print(f"Test:      {len(test):,} rows, columns: {list(test.columns)}")
    print(f"Reference: {len(reference):,} rows, columns: {list(reference.columns)}")

    train["Literal_clean"] = train["Literal"].apply(basic_clean)
    train["Literal_norm"] = train["Literal"].apply(normalize)

    train["code_type"] = train["Code"].astype(str).str[0].apply(
        lambda c: "diagnosis" if c.isalpha() else "procedure"
    )
    train["chapter"] = train["Code"].astype(str).str[0]
    train["is_icd10"] = train["Code"].isin(set(reference["Code"].unique()))

    train = train.merge(
        reference[["Code", "D_P", "Description"]],
        on="Code",
        how="left",
    )

    test["Literal_clean"] = test["Literal"].apply(basic_clean)
    test["Literal_norm"] = test["Literal"].apply(normalize)
    reference["Description_norm"] = reference["Description"].apply(normalize)

    print("Training data ready:")
    print(f"  {len(train):,} rows, {len(train.columns)} columns")
    print(
        f"  Diagnosis: {(train.code_type == 'diagnosis').sum():,}, "
        f"Procedure: {(train.code_type == 'procedure').sum():,}"
    )
    print(f"  ICD-10:    {train.is_icd10.sum():,}, ICD-9: {(~train.is_icd10).sum():,}")
    print(f"  Unique codes: {train.Code.nunique():,}")

    train.to_csv(DATA_PROCESSED_DIR / "train_preprocessed.csv", index=False)
    test.to_csv(DATA_PROCESSED_DIR / "test_preprocessed.csv", index=False)
    reference.to_csv(DATA_PROCESSED_DIR / "reference_preprocessed.csv", index=False)

    print("Saved:")
    print("  train_preprocessed.csv")
    print("  test_preprocessed.csv")
    print("  reference_preprocessed.csv")


if __name__ == "__main__":
    run()


## Preprocessing Runner

Former command-line wrapper for the preprocessing step.

_Archived from `scripts/step0_preprocessing.py`._


In [ ]:
from _bootstrap import PROJECT_ROOT  # noqa: F401
from icd10_coding.processing.preprocess import run


if __name__ == "__main__":
    run()
